In [0]:
display(dbutils.fs.ls("/Volumes/workspace/saas_revenue/saas"))


path,name,size,modificationTime
dbfs:/Volumes/workspace/saas_revenue/saas/ravenstack_accounts.csv,ravenstack_accounts.csv,36649,1770819532000
dbfs:/Volumes/workspace/saas_revenue/saas/ravenstack_churn_events.csv,ravenstack_churn_events.csv,44630,1770819532000
dbfs:/Volumes/workspace/saas_revenue/saas/ravenstack_feature_usage.csv,ravenstack_feature_usage.csv,1400898,1770819534000
dbfs:/Volumes/workspace/saas_revenue/saas/ravenstack_subscriptions.csv,ravenstack_subscriptions.csv,437566,1770819533000
dbfs:/Volumes/workspace/saas_revenue/saas/ravenstack_support_tickets.csv,ravenstack_support_tickets.csv,145598,1770819532000


In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
base_path = "/Volumes/workspace/saas_revenue/saas"

In [0]:
def ingest_csv_bronze(file_name,table_name):
    df = (
        spark.read
            .option("header",True)
            .option("inferSchema",True)
            .csv(f"{base_path}/{file_name}")
        )
    df = df.withColumn("injestion_ts",current_timestamp())
    (
        df.write
        .mode("append")
        .format("delta")
        .saveAsTable(table_name)
    )
def ingest_csv_silver(file_name,table_name):
    df = (
        spark.read
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_name)
    )

    

In [0]:
tables = [
    ("ravenstack_accounts.csv", "bronze_accounts"),
    ("ravenstack_churn_events.csv", "bronze_churn_events"),
    ("ravenstack_feature_usage.csv", "bronze_feature_usage"),
    ("ravenstack_subscriptions.csv", "bronze_subscriptions"),
    ("ravenstack_support_tickets.csv", "bronze_support_tickets"),
]


In [0]:
for file_name, table_name in tables:
    ingest_csv_bronze(file_name, table_name)

In [0]:
%sql
SHOW TABLES;


database,tableName,isTemporary
default,bronze_accounts,false
default,bronze_churn_events,false
default,bronze_feature_usage,false
default,bronze_subscriptions,false
default,bronze_support_tickets,false
default,dates,false
default,dim_date,false
default,dim_product,false
default,dim_session,false
default,fact_refunds,false


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.analytics;


In [0]:
%sql
SELECT * FROM workspace.default.bronze_accounts LIMIT 10;


account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag,ingestion_ts
A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,false,false,2026-02-11T15:13:05.801Z
A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,false,true,2026-02-11T15:13:05.801Z
A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,false,false,2026-02-11T15:13:05.801Z
A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,true,false,2026-02-11T15:13:05.801Z
A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,false,true,2026-02-11T15:13:05.801Z
A-1b9609,Company_5,EdTech,IN,2023-10-12,ads,Enterprise,4,false,false,2026-02-11T15:13:05.801Z
A-a0ca4e,Company_6,Cybersecurity,US,2024-03-08,ads,Pro,11,false,false,2026-02-11T15:13:05.801Z
A-e5d6ab,Company_7,EdTech,US,2023-04-15,partner,Pro,3,false,false,2026-02-11T15:13:05.801Z
A-7dacce,Company_8,Cybersecurity,CA,2024-09-10,event,Enterprise,12,false,true,2026-02-11T15:13:05.801Z
A-10b8da,Company_9,DevTools,US,2023-05-08,partner,Enterprise,14,false,true,2026-02-11T15:13:05.801Z


In [0]:
%sql
SHOW TABLES IN workspace.default;


database,tableName,isTemporary
default,bronze_accounts,false
default,bronze_churn_events,false
default,bronze_feature_usage,false
default,bronze_subscriptions,false
default,bronze_support_tickets,false
default,dates,false
default,dim_date,false
default,dim_product,false
default,dim_session,false
default,fact_refunds,false


In [0]:
%sql
SHOW TABLES IN hive_metastore.default;


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6692793678207556>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SHOW TABLES IN hive_metastore.default;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:192, in SqlMagic.sql(self, line, cell)
    186 except BaseException as e:
    187     self.driver_activity_logger

In [0]:
%sql
DESCRIBE workspace.default.bronze_accounts;


col_name,data_type,comment
account_id,string,null
account_name,string,null
industry,string,null
country,string,null
signup_date,date,null
referral_source,string,null
plan_tier,string,null
seats,int,null
is_trial,boolean,null
churn_flag,boolean,null


In [0]:
%sql
SELECT current_schema();

current_schema()
default


In [0]:
%sql
select account_id, count(*)
from workspace.default.bronze_accounts
group by account_id
having count(*) > 1;


account_id,count(*)


In [0]:
%sql
DESCRIBE workspace.default.bronze_subscriptions;


col_name,data_type,comment
subscription_id,string,null
account_id,string,null
start_date,date,null
end_date,date,null
plan_tier,string,null
seats,int,null
mrr_amount,int,null
arr_amount,int,null
is_trial,boolean,null
upgrade_flag,boolean,null


In [0]:
%sql
select subscription_id, count(*)
from workspace.default.bronze_subscriptions
group by subscription_id
having count(*) > 1;


subscription_id,count(*)
